# Chapter 3: Missing Data in Pandas

## Learning Objectives

In this chapter, we learn how to identify, inspect, remove, and handle missing values in time-series datasets using Pandas.

## Topics Covered

- Detecting missing values with `isna()` and `isnull()`
- Counting missing values by column and across a DataFrame
- Removing missing observations using `dropna()`
- Replacing missing values using `fillna()`
- Imputing missing values using mean, median, and other statistics
- Forward filling with `ffill()` and backward filling with `bfill()`
- Interpolating missing values using `interpolate()`
- Understanding edge cases when missing values occur at the beginning or end of a series
- Distinguishing missing values from missing timestamps
- Understanding data leakage when handling time-series data

## Why This Matters

Real-world datasets often contain missing measurements due to sensor failures, interrupted data collection, or recording errors.

In chemical engineering, missing observations might occur in reactor temperature, production rate, pressure, pH, or coating performance measurements.

Choosing an appropriate missing-data strategy is important because careless removal or imputation can distort the underlying process, introduce bias, or compromise a machine-learning model.

**Important:** Missing values should not be filled blindly. The method should depend on the meaning of the variable, the pattern of missingness, the time intervals, and the purpose of the analysis.

In [1]:

import pandas as pd
import numpy as np

dates = pd.date_range(
    start="2026-01-01",
    periods=8,
    freq="D"
)

df = pd.DataFrame({
    "temperature": [25, 26, np.nan, 28, 29, np.nan, 31, 32],
    "production": [100, 110, 105, np.nan, 125, 130, np.nan, 150],
    "pH": [7.0, 7.1, 7.2, 7.2, np.nan, 7.4, 7.5, 7.6]
}, index=dates)

df.index.name = "date"

df


,temperature,production,pH
date,,,
2026-01-01,25.0,100.0,7.0
2026-01-02,26.0,110.0,7.1
2026-01-03,NaN,105.0,7.2
2026-01-04,28.0,NaN,7.2
2026-01-05,29.0,125.0,NaN
2026-01-06,NaN,130.0,7.4
2026-01-07,31.0,NaN,7.5
2026-01-08,32.0,150.0,7.6


In [3]:
df.isna().sum()

,0
temperature,2
production,2
pH,1


In [4]:
# count the total number of missing cells in the entire dataset
df.isna().sum().sum()

np.int64(5)

In [5]:
# To check if the dataset contains any missing data
df.isna().any().any()

np.True_

In [7]:
df.production.isna().any()

np.True_


## Missing Data: Quick Reference

### Detecting and Counting Missing Values

| **Question** | **Code** |
|---|---|
| Which values are missing? | `df.isna()` |
| How many are missing per column? | `df.isna().sum()` |
| How many are missing in total? | `df.isna().sum().sum()` |
| Does a specific column contain any missing values? | `df["production"].isna().any()` |
| Does the entire DataFrame contain any missing values? | `df.isna().any().any()` |



## Dropping Values
 - dropna() removes any row that contains at least one missing value.
 - axis=0 means operate along rows. This is the default.
 - axis=1 means operate along columns.
 - df.dropna(how="all"). This drops only when all the values in that row are missing
 - df.dropna(subset=["temperature"]). This removes rows where temperature is missing, while allowing missing values in other columns.

## Filling missing values with fillna()
- df.fillna(0), this replaces every missing value in the DataFrame with zero.
However, filling with zero is only appropriate when zero is a meaningful value for the variable. A missing pH measurement does not mean pH was zero, and a missing temperature does not mean the reactor was at zero degrees.
- df["temperature"].fillna(df["temperature"].mean()), This fills a single column


In [8]:
df.fillna({
    "temperature": df["temperature"].mean(),
    "production": df["production"].median(),
    "pH": df["pH"].mean()
})

,temperature,production,pH
date,,,
2026-01-01,25.0,100.0,7.000000
2026-01-02,26.0,110.0,7.100000
2026-01-03,28.5,105.0,7.200000
2026-01-04,28.0,117.5,7.200000
2026-01-05,29.0,125.0,7.285714
2026-01-06,28.5,130.0,7.400000
2026-01-07,31.0,117.5,7.500000
2026-01-08,32.0,150.0,7.600000


These are examples of possible strategies, not universal rules. The median can be useful when data contains extreme values, while the mean is sensitive to them.
Also, if you're preparing data for machine learning, calculate imputation values from the training data only. Calculating them from the full dataset before splitting can leak information from the test set into training.

## ffill() and bfill()
- Forward filling carries the last observed value forward.
- Backward filling uses the next valid observation to fill missing values.

In forecasting, be careful with bfill(): using a future observation to fill an earlier missing value can introduce data leakage if you're preparing historical data to predict the future.

In [9]:
# Interpolation
s = pd.Series([10, np.nan, 14])
s.interpolate()

,0
0,10.0
1,12.0
2,14.0


The assumption is a linear change between the known observations.

In [10]:
s = pd.Series([10, np.nan, np.nan, 16])
s.interpolate()

,0
0,10.0
1,12.0
2,14.0
3,16.0


In [12]:
import pandas as pd
import numpy as np

s = pd.Series([np.nan, 10, np.nan, 14, np.nan])
print("Original series:")
print(s)

print("Forward fill:")
print(s.ffill())

print("\nBackward fill:")
print(s.bfill())

print("\nInterpolation:")
print(s.interpolate())

Original series:
0     NaN
1    10.0
2     NaN
3    14.0
4     NaN
dtype: float64
Forward fill:
0     NaN
1    10.0
2    10.0
3    14.0
4    14.0
dtype: float64

Backward fill:
0    10.0
1    10.0
2    14.0
3    14.0
4     NaN
dtype: float64

Interpolation:
0     NaN
1    10.0
2    12.0
3    14.0
4    14.0
dtype: float64


# Chapter 3: Summary and Knowledge Check

## 1. Summary

### Detecting Missing Values

- `df.isna()` returns a Boolean DataFrame indicating missing values.
- `df.isna().sum()` counts missing values in each column.
- `df.isna().sum().sum()` counts all missing cells.
- `df.isna().any().any()` checks whether any missing value exists in the DataFrame.

### Removing Missing Data

- `df.dropna()` removes rows containing at least one missing value by default.
- `df.dropna(axis=1)` removes columns containing missing values.
- `df.dropna(how="all")` removes rows where every value is missing.
- `df.dropna(subset=["temperature"])` removes rows where the specified column is missing.

Removing incomplete rows can discard useful observations and potentially introduce bias.

### Filling Missing Values

- `fillna(0)` replaces missing values with zero, which is appropriate only when zero has a meaningful interpretation.
- Mean or median imputation replaces missing values with a column statistic.
- Mean imputation is sensitive to extreme values, while median imputation is generally more robust to outliers.

### Forward Fill and Backward Fill

- `ffill()` uses the previous valid observation.
- `bfill()` uses the next valid observation.
- Forward filling may be suitable when the previous measurement remains representative for a short period.
- Backward filling can introduce future information into forecasting workflows.

### Interpolation

- `interpolate()` estimates missing values between known observations.
- Linear interpolation estimates values along a straight line.
- For irregularly spaced timestamps, `interpolate(method="time")` can account for elapsed time when using a `DatetimeIndex`.
- By default, linear interpolation generally leaves missing values at the boundaries unfilled.

Interpolation is an estimate, not proof of the actual missing measurement.

### Missing Values vs. Missing Timestamps

- A missing value occurs when a timestamp exists but one or more measurements are unavailable.
- A missing timestamp occurs when an expected time point is absent from the index.
- `asfreq()` can expose missing timestamps when the expected frequency is known.

### Data Leakage

Data leakage occurs when a model uses information that would not legitimately be available at prediction time.

Using backward filling or other preprocessing methods that expose future observations can compromise a forecasting model.

## 2. Knowledge Check

1. What is the difference between `isna()` and `isna().sum()`?
2. What is the difference between `dropna()` and `fillna()`?
3. Why might replacing every missing value with zero be inappropriate?
4. When would forward filling be reasonable for a time-series dataset?
5. How does backward filling differ from forward filling?
6. How does linear interpolation estimate missing values between two known observations?
7. Why might interpolation be inappropriate for a process with sudden changes or spikes?
8. What is the difference between a missing value and a missing timestamp?
9. Why can dropping all incomplete rows introduce bias?
10. What is data leakage, and why can backward filling be problematic in forecasting?

## 3. Key Takeaway

There is no universally correct method for handling missing data. Inspect the missingness pattern, understand the process that generated the data, select a suitable method, and ensure that preprocessing does not introduce information unavailable at prediction time.